In [0]:
from pyspark.sql import functions as F

In [0]:
from pyspark.sql.types import *

In [0]:
schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("user_id", StringType(), False),
    StructField("amount", DoubleType(), False),
    StructField("currency", StringType(), False),
    StructField("merchant", StringType(), False),
    StructField("category", StringType(), False),
    StructField("payment_method", StringType(), False),
    StructField("status", StringType(), False),
    StructField("device_os", StringType(), False),
    StructField("extraction_timestamp", TimestampType(), False)
])

In [0]:
df = spark.range(10000)

df = df.withColumnRenamed("id", "transaction_number")

display(df.limit(10))

In [0]:
df = df.withColumn(
    "transaction_id",
    F.concat(
        F.lit("TXN-"),
        F.lpad(F.col("transaction_number").cast("string"), 8, "0")
    )
)

display(df.limit(10))

In [0]:
df = df.withColumn(
    "user_id",
    F.concat(
        F.lit("USR-"),
        F.lpad(
            (F.floor(F.rand(seed=42) * 2000) + 1).cast("string"),
            5,
            "0"
        )
    )
)

display(df.limit(10))

In [0]:
df = df.withColumn(
    "amount",
    F.round(
        F.rand(seed=100) * 990 + 10,
        2
    )
)

display(df.limit(10))

In [0]:
df = df.withColumn(
    "currency",
    F.when(F.rand(seed=200) < 0.85, "BRL")
     .when(F.rand(seed=201) < 0.10, "USD")
     .otherwise("EUR")
)

display(df.limit(10))

In [0]:
df = df.withColumn(
    "merchant",
    F.element_at(
        F.array(
            F.lit("Amazon"),
            F.lit("Mercado Livre"),
            F.lit("Magazine Luiza"),
            F.lit("iFood"),
            F.lit("Uber"),
            F.lit("Netflix"),
            F.lit("Spotify"),
            F.lit("Drogasil")
        ),
        (F.floor(F.rand(seed=300) * 8) + 1).cast("int")
    )
)

display(df.limit(10))

In [0]:
df = df.withColumn(
    "category",
    F.when(F.rand(seed=400) < 0.20, "Eletronicos")
     .when(F.rand(seed=401) < 0.20, "Alimentacao")
     .when(F.rand(seed=402) < 0.15, "Transporte")
     .when(F.rand(seed=403) < 0.15, "Entretenimento")
     .when(F.rand(seed=404) < 0.10, "Farmacia")
     .when(F.rand(seed=405) < 0.10, "Vestuaria")
     .otherwise("Outros")
)

display(df.limit(10))


In [0]:
df = df.withColumn(
    "payment_method",
    F.when(F.rand(seed=500) < 0.40, "Cartao_Credito")
     .when(F.rand(seed=501) < 0.30, "Pix")
     .when(F.rand(seed=502) < 0.15, "Cartao_Debito")
     .when(F.rand(seed=503) < 0.10, "Boleto")
     .otherwise("Carteira_Digital")
)

display(df.limit(10))


In [0]:
df = df.withColumn(
    "status",
    F.when(F.rand(seed=600) < 0.75, "Aprovado")
     .when(F.rand(seed=601) < 0.15, "Recusado")
     .when(F.rand(seed=602) < 0.07, "Pendente")
     .otherwise("Suspeita_Fraude")
)

display(df.limit(10))

In [0]:
df = df.withColumn(
    "device_os",
    F.when(F.rand(seed=700) < 0.55, "Android")
     .when(F.rand(seed=701) < 0.35, "iOS")
     .otherwise("Windows")
)

display(df.limit(10))


In [0]:
df = df.withColumn(
    "extraction_timestamp",
    F.current_timestamp()
)

display(df.limit(10))


In [0]:
df.printSchema()

In [0]:
print("Total de registros:", df.count())


In [0]:
null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

display(null_counts)

In [0]:
print("=== VALIDAÇÃO DA BRONZE ===")

# 1. Quantidade de registros
total = df.count()
print(f"Total de registros: {total}")

# 2. Quantidade de transaction_id distintos
unique_transactions = df.select("transaction_id").distinct().count()
print(f"Transaction IDs distintos: {unique_transactions}")

# 3. Duplicidades
duplicates = total - unique_transactions
print(f"Transaction IDs duplicados: {duplicates}")

# 4. Valores de amount
df.select(
    F.min("amount").alias("valor_minimo"),
    F.max("amount").alias("valor_maximo"),
    F.avg("amount").alias("ticket_medio")
).show()

# 5. Distribuição dos status
print("=== STATUS ===")
df.groupBy("status").count().orderBy(F.desc("count")).show()

# 6. Distribuição dos métodos de pagamento
print("=== MÉTODOS DE PAGAMENTO ===")
df.groupBy("payment_method").count().orderBy(F.desc("count")).show()

# 7. Distribuição dos sistemas operacionais
print("=== SISTEMAS OPERACIONAIS ===")
df.groupBy("device_os").count().orderBy(F.desc("count")).show()

In [0]:
df_bronze = df.select(
    "transaction_id",
    "user_id",
    "amount",
    "currency",
    "merchant",
    "category",
    "payment_method",
    "status",
    "device_os",
    "extraction_timestamp"
)

display(df_bronze.limit(10))

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
SHOW SCHEMAS IN workspace;

In [0]:
%sql

CREATE VOLUME IF NOT EXISTS workspace.default.financial_lake;

In [0]:
bronze_path = "/Volumes/workspace/default/financial_lake/bronze/transactions"

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .save(bronze_path)

In [0]:
df_bronze_check = spark.read.format("delta").load(bronze_path)

print("Registros gravados:", df_bronze_check.count())

In [0]:
print("=== BRONZE ===")

df_bronze_check.printSchema()

print(f"Total de registros: {df_bronze_check.count()}")

print("=== AMOSTRA ===")
display(df_bronze_check.limit(10))
